# Analysis of my Japanese Immersion

## Introduction

This is an analysis of the things I've watched/read 100% on Japanese. I started dabbling with Japanese content in late June 2023, but began
formally studying and immersing on July 3rd, 2023 — the date I consider my real starting point. I started immersing in the language at a young stage. Almost the entirety of the things I've immersed on Japanese is tracked here, I made sure to track almost every hour.

Below you'll find my analysis of this data, how I've improved, my ups and downs and specially we'll find if learning is linear or not.

This data was taken from nihongotracker.app, a website developed to track your Japanese immersion.

## Data Cleaning

Before fixing anything, I'll run a systematic diagnostic pass over the whole dataset — general structure, missing values, duplicates, and numeric ranges. This is what actually tells me where to look, instead of patching issues as I happen to notice them.

In [ ]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1824 entries, 0 to 1823
Data columns (total 9 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   date         1824 non-null   object 
 1   type         1824 non-null   object 
 2   mediaId      1771 non-null   object 
 3   time         1332 non-null   float64
 4   characters   1046 non-null   float64
 5   episodes     1089 non-null   float64
 6   pages        568 non-null    float64
 7   description  1824 non-null   object 
 8   tags         21 non-null     object 
dtypes: float64(4), object(5)
memory usage: 128.4 KB


**Finding:** `date`, `type`, and `description` are fully populated (1824/1824). `date` is still stored as `object` (string), not `datetime` — needs converting before any time-based analysis. `time`, `characters`, `episodes`, and `pages` all have large gaps, but that alone doesn't mean they're broken — different media types (anime, manga, vn, etc.) naturally fill different columns. That's checked separately below, by media type.

In [15]:
df.isnull().sum()

date              0
type              0
mediaId          53
time            492
characters      778
episodes        735
pages          1256
description       0
tags           1803
dtype: int64

**Finding:** It's pretty normal that `time`, `characters`, `episodes` and `pages` are mostly null because those are dependable data, it depends on which media type are you logging. `mediaId` depends on anilist, vndb and TMDB, nihongotracker match the things I log using those sites so if media isn't there, mediaId will be null. And to finish with, `tags` is mostly null because I only use it if I want to make some specification.

In [17]:
df.duplicated().sum
df[df.duplicated(keep=False)]

,date,type,mediaId,time,characters,episodes,pages,description,tags
26,2026-07-28,anime,207141,24.0,0.0,1.0,0.0,ヤニねこ,NaN
27,2026-07-28,anime,207141,24.0,0.0,1.0,0.0,ヤニねこ,NaN
34,2026-07-23,video,UCdFukkLFB6GOsfAAkNnmZJA,1.0,0.0,0.0,0.0,放送前は全く知らなかったこの作品が予想を超えてくるとんでもないアニメでした...【いびってこ...,NaN
35,2026-07-23,video,UCdFukkLFB6GOsfAAkNnmZJA,1.0,0.0,0.0,0.0,放送前は全く知らなかったこの作品が予想を超えてくるとんでもないアニメでした...【いびってこ...,NaN
36,2026-07-23,video,UChAQQ9at2MDlXjvQX3vL3Yw,1.0,0.0,0.0,0.0,【旅行Vlog】出発前日に航空券とって長崎旅行に来た人の無計画旅行Vlog,NaN
...,...,...,...,...,...,...,...,...,...
1567,2024-02-21,anime,101573,NaN,NaN,1.0,NaN,yagate kimi ni naru,NaN
1583,2024-02-07,anime,101573,NaN,NaN,1.0,NaN,yagate kimi ni naru,NaN
1584,2024-02-07,anime,101573,NaN,NaN,1.0,NaN,yagate kimi ni naru,NaN
1744,2023-10-22,anime,130003,NaN,NaN,1.0,NaN,bocchi,NaN


**Finding:** Most of these duplicated data is due to logging the same thing the same day, for example, if I watched an episode of some anime, logged it and later, the same day, watch another and log it again, it counts it as duplicate without necessarily being a duplicate, so I'll leave it.

In [18]:
df.describe()

,time,characters,episodes,pages
count,1332.00000,1046.000000,1089.000000,568.000000
mean,73.48048,9097.745698,1.469238,25.551056
std,93.71895,13182.405655,3.013026,60.295166
min,0.00000,0.000000,0.000000,0.000000
25%,22.00000,82.750000,0.000000,0.000000
50%,50.00000,4882.500000,1.000000,0.000000
75%,94.00000,11708.000000,1.000000,0.000000
max,1567.00000,161025.000000,57.000000,241.000000


**Finding:** Looks fine, I also used to log more than 1 day at once, for example when it came to books because I wanted to log the entire book once I finished it so it's normal to see more than it's possible on one day, that's way I prefer to base it monthly instead of daily.

### Checking consistency by media typeWith the overall diagnostic done, the next step is checking whether the missing values above are structural (a manga entry has no `episodes` because manga doesn't have episodes) or a real data problem — starting with the per-type breakdown.

In [11]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

df = pd.read_csv("../data/nihongotracker-export.csv", on_bad_lines="skip")

# check consistency of data in every media type
print(df.groupby('type')[['time', 'characters', 'episodes', 'pages']].count())

# check why there are episodes on some movie media type
print(df[(df['type']=='movie') & (df['episodes'].notnull())])

             time  characters  episodes  pages
type                                          
anime          82          54       539     85
audio          13           1         7      2
light-novel   157         133         8      7
manga         120          86        16    121
movie          35           0         9      0
reading        19          19         8      8
tv show        69           1        19      5
video         313         200       284    243
vn            524         552       199     97
            date   type   mediaId   time  characters  episodes  pages  \
126   2026-06-11  movie    m68341  146.0         NaN       1.0    NaN   
321   2026-05-11  movie    m36917   60.0         NaN       1.0    NaN   
406   2026-02-22  movie     m9666   90.0         NaN       1.0    NaN   
411   2026-02-19  movie    m11838   90.0         NaN       1.0    NaN   
412   2026-02-17  movie    m32250  110.0         NaN       1.0    NaN   
450   2026-01-27  movie  m1425714   97.0     

It seems there are some kind of mistake when some movies where logged, some of them have data on the episodes field. Since the number of episodes is a clear number I can conclude is a mistake and it's due for the following reason:

**Some movies where logged with 1 episode, assuming 1 movie counted as 1 episode.**

In [12]:
#remove episodes from movies due to a mistake
df.loc[df['type'] == 'movie', 'episodes'] = None

print(df[(df['type']=='movie') & (df['episodes'].notnull())])

Empty DataFrame
Columns: [date, type, mediaId, time, characters, episodes, pages, description, tags]
Index: []
